# Neural Trajectories in Attempted Speech

Run alignment-aware trajectory analyses on the durable Willett representation exports created by `s11_willett_representation_manifolds.ipynb`. The notebook reads and writes Drive-backed artifacts, so checkpoint extraction does not need to be repeated.

CTC alignment is model-assisted timing. Hidden-state paths describe decoder dynamics; `input_windows` are a closer neural-input control but are still overlapping 280 ms windows at 80 ms cadence.

In [ ]:
# Colab / Drive / repository bootstrap.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Not running in Colab or Drive already unavailable: {exc}')

import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/utah-ssl') if Path('/content').exists() else Path.cwd()
REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'
if str(REPO_DIR).startswith('/content'):
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(REPO_DIR), check=False)
os.chdir(REPO_DIR)
PACKAGE_ROOT = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments'
os.environ['PYTHONPATH'] = f"{PACKAGE_ROOT}:{os.environ.get('PYTHONPATH', '')}"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

try:
    import sklearn, matplotlib, pandas  # noqa: F401
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'matplotlib', 'scikit-learn'], check=True)
print('REPO_DIR:', REPO_DIR)
print('PACKAGE_ROOT:', PACKAGE_ROOT)

## Configuration

By default this targets the released Stanford GRU export configured in `s11`. Change `EXPORT_NAME` or `MODEL_KEY` to analyze another saved model.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else Path('/Users/home/My Drive')
UTAH_SSL_ROOT = DRIVE_ROOT / 'utah_ssl'
REPRESENTATION_ROOT = UTAH_SSL_ROOT / 'data' / 'representations' / 'willett_manifolds'
OUTPUT_ROOT = UTAH_SSL_ROOT / 'outputs' / 'neural_trajectories'

EXPORT_NAME = 'stanford_released_gru_s5_val_area6v_soft_phonetic_categories_v1'
MODEL_KEY = 'gru_released'
MODEL_DIR = REPRESENTATION_ROOT / EXPORT_NAME / MODEL_KEY

# Hidden is always present. Input controls require an s11 export made with
# SAVE_INPUT_WINDOWS=True. Missing optional controls are skipped cleanly.
REPRESENTATIONS = ('hidden', 'input_windows', 'adapted_input_windows')
MIN_TRIALS = 20
MAX_EVENTS_PER_PHONEME = 100
BEFORE = 3
AFTER = 3
COMPONENTS = 6
PERMUTATIONS = 1000
RELIABILITY_REPETITIONS = 200
SEED = 7

print('MODEL_DIR:', MODEL_DIR)
if not (MODEL_DIR / 'metadata.json').exists():
    available = sorted(str(path.relative_to(REPRESENTATION_ROOT)) for path in REPRESENTATION_ROOT.glob('*/*/metadata.json')) if REPRESENTATION_ROOT.exists() else []
    raise FileNotFoundError(
        f'No representation export found at {MODEL_DIR}. Run the s11 export cells first '        f'or change EXPORT_NAME/MODEL_KEY. Available exports: {available}'
    )

## Run trajectory analyses

Each representation gets its own durable output folder. Existing outputs are overwritten by a fresh deterministic run.

In [ ]:
import json
import numpy as np

shard_manifest = json.loads((MODEL_DIR / 'shards.json').read_text())
with np.load(MODEL_DIR / 'shards' / shard_manifest[0]['shard']) as first_shard:
    available_arrays = set(first_shard.files)
print('Available shard arrays:', sorted(available_arrays))

run_summaries = {}
for representation in REPRESENTATIONS:
    if representation not in available_arrays:
        print(f'\nSkipping {representation}: it was not saved by s11.')
        continue
    output_dir = OUTPUT_ROOT / EXPORT_NAME / MODEL_KEY / representation
    command = [
        sys.executable, '-m', 'neural_trajectories.run_export_analysis',
        str(MODEL_DIR), str(output_dir),
        '--representation', representation,
        '--min-trials', str(MIN_TRIALS),
        '--before', str(BEFORE),
        '--after', str(AFTER),
        '--max-events-per-phoneme', str(MAX_EVENTS_PER_PHONEME),
        '--components', str(COMPONENTS),
        '--permutations', str(PERMUTATIONS),
        '--reliability-repetitions', str(RELIABILITY_REPETITIONS),
        '--seed', str(SEED),
    ]
    print(f'\n=== {representation} ===')
    subprocess.run(command, cwd=str(REPO_DIR), check=True, env=os.environ.copy())
    run_summaries[representation] = json.loads((output_dir / 'summary.json').read_text())

print(json.dumps(run_summaries, indent=2))

## Display results

The red point marks the CTC-aligned phoneme center. Thin gray lines are individual occurrences and the blue line is their mean path.

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary_rows = []
for representation, summary in run_summaries.items():
    separation = summary['separation']
    summary_rows.append({
        'representation': representation,
        'events': summary['retained_event_count'],
        'phonemes': summary['retained_phoneme_count'],
        'between_minus_within': separation['between_minus_within'],
        'permutation_p': separation['permutation_p_value'],
        'PC_variance_sum': sum(summary['pca_explained_variance_ratio']),
    })
display(pd.DataFrame(summary_rows))

for representation in run_summaries:
    output_dir = OUTPUT_ROOT / EXPORT_NAME / MODEL_KEY / representation
    print(f'\n{representation}: {output_dir}')
    display(Image(filename=str(output_dir / 'phoneme_trajectories.png')))
    repeatability = pd.read_csv(output_dir / 'phoneme_repeatability.csv')
    display(repeatability.sort_values('split_half_reliability', ascending=False).reset_index(drop=True))

## Reading the result

A useful hidden-state result requires both positive path-shape separation and reliable individual phonemes. Compare it with `input_windows`: hidden-only structure supports a decoder-representation claim, while matching input-window structure is closer to evidence in the neural measurements. Neither input-window control is truly raw because each point contains an overlapping 280 ms window.

The next strict test is a pre-patching 20 ms raw-bin export, followed by session-stratified and leave-session-out analysis.

In [ ]:
# Optional Colab resource release.
try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass